# EXP-010 | DINOv2 + estadísticas por plano + MLP multietiqueta

**Desarrollo propio:** conserva el brazo EXP-009 y prueba una rama no lineal sobre las mismas inferencias de DINOv2. Usa 4,349 weak y 58 gold para auditoria. Antes de enviar mira **RECOMENDACION SUBMIT**.

El notebook se generó de `src/exp009_dinov2_weaklabels.py` y `src/exp010_dinov2_multitask.py`. Requiere **GPU T4**, Internet OFF y los mismos tres inputs de EXP-009. No garantiza 0.80 en Kaggle.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import pydicom
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

UID = "StudyInstanceUID"
SID = "SeriesInstanceUID"

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

PLANES = [
    "sagittal",
    "coronal",
    "axial",
]

SERIES_PER_PLANE = 2
WINDOWS_PER_SERIES = 4
SEED = 42


def find_data():
    candidates = [
        Path(
            "/kaggle/input/competitions/"
            "rsna-knee-abnormality-detection"
        ),
        Path(
            "/kaggle/input/"
            "rsna-knee-abnormality-detection"
        ),
    ]

    for folder in candidates:
        if (
            (folder / "train.csv").exists()
            and
            (folder / "train_series").exists()
        ):
            return folder

    raise FileNotFoundError(
        "No encuentro los datos de la competencia."
    )


def find_dino():
    candidates = [
        Path(
            "/kaggle/input/dinov2/"
            "pytorch/small/1"
        ),
        Path(
            "/kaggle/input/models/metaresearch/"
            "dinov2/pytorch/small/1"
        ),
    ]

    for folder in candidates:
        if (
            folder / "config.json"
        ).exists():
            return folder

    for config in Path(
        "/kaggle/input"
    ).rglob("config.json"):

        text = str(config).lower()

        if (
            "dinov2" in text
            and "small" in text
        ):
            return config.parent

    raise FileNotFoundError(
        "Agrega Meta DINO v2 > "
        "PyTorch > small como Input."
    )


def boolish(value):
    if pd.isna(value):
        return 0

    return int(
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
            "y",
            "t",
        }
    )


def normalize_plane(value):
    text = str(value).lower()

    for plane in PLANES:
        if plane in text:
            return plane

    return "other"


def choose_series(series_df):
    df = series_df.copy()

    df["_plane"] = (
        df["Anatomical_Plane"]
        .map(normalize_plane)
    )

    df["_fluid"] = (
        df["Fluid_Sensitive"]
        .map(boolish)
    )

    df["_fat"] = (
        df["Fat_Suppression"]
        .map(boolish)
    )

    df["_score"] = (
        2 * df["_fluid"]
        + df["_fat"]
    )

    selected = {}

    for study_uid, group in df.groupby(
        UID,
        sort=False,
    ):
        selected[
            str(study_uid)
        ] = {}

        for plane in PLANES:
            gp = (
                group[
                    group["_plane"]
                    == plane
                ]
                .sort_values(
                    "_score",
                    ascending=False,
                    kind="stable",
                )
            )

            chosen = []
            protocols = set()

            for _, row in gp.iterrows():

                protocol = (
                    int(row["_fluid"]),
                    int(row["_fat"]),
                )

                series_uid = str(
                    row[SID]
                )

                if (
                    protocol
                    not in protocols
                ):
                    chosen.append(
                        series_uid
                    )

                    protocols.add(
                        protocol
                    )

                if (
                    len(chosen)
                    == SERIES_PER_PLANE
                ):
                    break

            if (
                len(chosen)
                < SERIES_PER_PLANE
            ):
                for series_uid in (
                    gp[SID]
                    .astype(str)
                ):
                    if (
                        series_uid
                        not in chosen
                    ):
                        chosen.append(
                            series_uid
                        )

                    if (
                        len(chosen)
                        == SERIES_PER_PLANE
                    ):
                        break

            selected[
                str(study_uid)
            ][plane] = chosen

    return selected


def sorted_files(series_dir):
    records = []

    for order, path in enumerate(
        series_dir.glob("*.dcm")
    ):

        try:
            ds = pydicom.dcmread(
                path,
                stop_before_pixels=True,
                force=True,
                specific_tags=[
                    "ImagePositionPatient",
                    "ImageOrientationPatient",
                    "InstanceNumber",
                ],
            )

            ipp = getattr(
                ds,
                "ImagePositionPatient",
                None,
            )

            iop = getattr(
                ds,
                "ImageOrientationPatient",
                None,
            )

            if (
                ipp is not None
                and iop is not None
                and len(iop) >= 6
            ):
                row = np.asarray(
                    iop[:3],
                    float,
                )

                col = np.asarray(
                    iop[3:6],
                    float,
                )

                normal = np.cross(
                    row,
                    col,
                )

                position = np.asarray(
                    ipp[:3],
                    float,
                )

                key = float(
                    np.dot(
                        position,
                        normal,
                    )
                )

            else:
                key = float(
                    getattr(
                        ds,
                        "InstanceNumber",
                        order,
                    )
                )

        except Exception:
            key = float(order)

        records.append(
            (
                key,
                path,
            )
        )

    records.sort(
        key=lambda value: value[0]
    )

    return [
        path
        for _, path
        in records
    ]


def read_u8(path):
    ds = pydicom.dcmread(
        path,
        force=True,
    )

    array = (
        ds.pixel_array
        .astype(np.float32)
    )

    array = (
        array
        * float(
            getattr(
                ds,
                "RescaleSlope",
                1.0,
            )
        )
        + float(
            getattr(
                ds,
                "RescaleIntercept",
                0.0,
            )
        )
    )

    if (
        str(
            getattr(
                ds,
                "PhotometricInterpretation",
                "",
            )
        ).upper()
        == "MONOCHROME1"
    ):
        array = (
            array.max()
            - array
        )

    valid = np.isfinite(
        array
    )

    if not valid.any():
        return np.zeros(
            array.shape,
            dtype=np.uint8,
        )

    low, high = np.percentile(
        array[valid],
        [
            1,
            99,
        ],
    )

    if high <= low:
        high = low + 1

    array = np.clip(
        (
            array
            - low
        )
        / (
            high
            - low
        ),
        0,
        1,
    )

    return (
        255 * array
    ).round().astype(
        np.uint8
    )


def make_triplet(
    files,
    center,
):
    indices = [
        max(
            0,
            center - 1,
        ),
        center,
        min(
            len(files) - 1,
            center + 1,
        ),
    ]

    channels = [
        read_u8(
            files[index]
        )
        for index
        in indices
    ]

    height = min(
        image.shape[0]
        for image in channels
    )

    width = min(
        image.shape[1]
        for image in channels
    )

    rgb = np.stack(
        [
            image[
                :height,
                :width,
            ]
            for image
            in channels
        ],
        axis=-1,
    )

    return Image.fromarray(
        rgb
    )


def series_windows(
    series_dir,
):
    files = sorted_files(
        series_dir
    )

    if len(files) < 2:
        return []

    centers = np.rint(
        np.linspace(
            0.15,
            0.85,
            WINDOWS_PER_SERIES,
        )
        * (
            len(files)
            - 1
        )
    ).astype(int)

    images = []

    for center in centers:
        try:
            images.append(
                make_triplet(
                    files,
                    int(center),
                )
            )

        except Exception:
            pass

    return images


@torch.inference_mode()
def encode(
    images,
    processor,
    model,
    device,
):
    if not images:
        return np.empty(
            (
                0,
                model.config.hidden_size,
            ),
            dtype=np.float32,
        )

    features = []

    for start in range(
        0,
        len(images),
        24,
    ):
        batch = images[
            start:
            start + 24
        ]

        pixel_values = (
            processor(
                images=batch,
                return_tensors="pt",
            )["pixel_values"]
            .to(device)
        )

        if (
            device.type
            == "cuda"
        ):
            with torch.autocast(
                "cuda",
                dtype=torch.float16,
            ):
                output = (
                    model(
                        pixel_values=
                        pixel_values
                    )
                    .last_hidden_state[
                        :,
                        0,
                    ]
                )

        else:
            output = (
                model(
                    pixel_values=
                    pixel_values
                )
                .last_hidden_state[
                    :,
                    0,
                ]
            )

        features.append(
            output
            .float()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        features
    )


def one_study(
    study_uid,
    chosen,
    dicom_root,
    processor,
    model,
    device,
):
    hidden = int(
        model.config.hidden_size
    )

    blocks = []

    for plane in PLANES:

        images = []

        for series_uid in (
            chosen
            .get(
                str(study_uid),
                {},
            )
            .get(
                plane,
                [],
            )
        ):

            series_dir = (
                dicom_root
                / str(study_uid)
                / str(series_uid)
            )

            if (
                series_dir.exists()
            ):
                images += (
                    series_windows(
                        series_dir
                    )
                )

        features = encode(
            images,
            processor,
            model,
            device,
        )

        if len(features):

            blocks.append(
                features.mean(
                    axis=0
                )
            )

            blocks.append(
                features.max(
                    axis=0
                )
            )

        else:

            blocks.append(
                np.zeros(
                    hidden,
                    np.float32,
                )
            )

            blocks.append(
                np.zeros(
                    hidden,
                    np.float32,
                )
            )

    return np.concatenate(
        blocks
    ).astype(
        np.float32
    )


def build_matrix(
    study_uids,
    chosen,
    root,
    processor,
    model,
    device,
    label,
):
    output = []
    times = []

    total = len(
        study_uids
    )

    for index, study_uid in enumerate(
        study_uids,
        start=1,
    ):

        start = time.time()

        output.append(
            one_study(
                study_uid,
                chosen,
                root,
                processor,
                model,
                device,
            )
        )

        times.append(
            time.time()
            - start
        )

        if (
            index % 10 == 0
            or index == total
        ):
            average = np.mean(
                times[
                    -min(
                        20,
                        len(times),
                    ):
                ]
            )

            print(
                f"{label}: "
                f"{index}/{total} | "
                f"{average:.2f} "
                "s/estudio"
            )

    return (
        np.vstack(
            output
        ),
        times,
    )


def candidates():
    return {

        "PCA16_C0.1":
        Pipeline(
            [
                (
                    "scale",
                    StandardScaler(),
                ),
                (
                    "pca",
                    PCA(
                        16,
                        whiten=True,
                        random_state=SEED,
                    ),
                ),
                (
                    "lr",
                    LogisticRegression(
                        C=0.1,
                        class_weight=
                        "balanced",
                        solver=
                        "liblinear",
                        max_iter=3000,
                        random_state=SEED,
                    ),
                ),
            ]
        ),

        "PCA24_C0.1":
        Pipeline(
            [
                (
                    "scale",
                    StandardScaler(),
                ),
                (
                    "pca",
                    PCA(
                        24,
                        whiten=True,
                        random_state=SEED,
                    ),
                ),
                (
                    "lr",
                    LogisticRegression(
                        C=0.1,
                        class_weight=
                        "balanced",
                        solver=
                        "liblinear",
                        max_iter=3000,
                        random_state=SEED,
                    ),
                ),
            ]
        ),

        "RAW_C0.002":
        Pipeline(
            [
                (
                    "scale",
                    StandardScaler(),
                ),
                (
                    "lr",
                    LogisticRegression(
                        C=0.002,
                        class_weight=
                        "balanced",
                        solver=
                        "liblinear",
                        max_iter=3000,
                        random_state=SEED,
                    ),
                ),
            ]
        ),
    }


def cv_score(
    pipeline,
    X,
    gold,
):
    scores = {}

    for target in TARGETS:

        y = (
            gold[target]
            .astype(int)
            .to_numpy()
        )

        minimum_class = int(
            pd.Series(y)
            .value_counts()
            .min()
        )

        cv = StratifiedKFold(
            n_splits=min(
                5,
                minimum_class,
            ),
            shuffle=True,
            random_state=SEED,
        )

        prediction = (
            cross_val_predict(
                pipeline,
                X,
                y,
                cv=cv,
                method=
                "predict_proba",
            )[
                :,
                1,
            ]
        )

        scores[target] = (
            roc_auc_score(
                y,
                prediction,
            )
        )

    macro = float(
        np.mean(
            list(
                scores.values()
            )
        )
    )

    return (
        macro,
        scores,
    )

def normalized_name(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def find_weak_labels():
    weak_path = Path(
        "/kaggle/input/datasets/"
        "stevenleehans/"
        "rsna-knee-llm-report-labels/"
        "llm_labels_v4_blend.csv"
    )

    if not weak_path.exists():
        raise FileNotFoundError(
            "No encuentro llm_labels_v4_blend.csv"
        )

    return weak_path

def find_uid_column(df):
    for column in df.columns:
        if (
            normalized_name(column)
            == "studyinstanceuid"
        ):
            return column

    raise KeyError(
        "No encuentro StudyInstanceUID "
        "en weak labels."
    )


def find_score_columns(df):
    mapping = {}

    for target in TARGETS:
        target_normalized = (
            normalized_name(target)
        )

        candidates = []

        for column in df.columns:
            column_normalized = (
                normalized_name(column)
            )

            if (
                target_normalized
                in column_normalized
                and "score"
                in column_normalized
            ):
                candidates.append(
                    column
                )

        # Fallback:
        # columna exactamente llamada como target.
        if not candidates:
            for column in df.columns:
                if (
                    normalized_name(
                        column
                    )
                    == target_normalized
                ):
                    candidates.append(
                        column
                    )

        if not candidates:
            raise KeyError(
                f"No encuentro score para {target}"
            )

        mapping[target] = (
            candidates[0]
        )

    return mapping


def rank01(values):
    return (
        pd.Series(
            np.asarray(values)
        )
        .rank(
            method="average",
            pct=True,
        )
        .to_numpy(
            dtype=float
        )
    )


def macro_auc(
    gold,
    predictions,
):
    scores = {}

    for target in TARGETS:
        scores[target] = (
            roc_auc_score(
                gold[target]
                .astype(int),
                predictions[target],
            )
        )

    return (
        float(
            np.mean(
                list(
                    scores.values()
                )
            )
        ),
        scores,
    )


def gold_oof_and_test(
    X_gold,
    X_test,
    gold,
):
    oof = {}
    test_predictions = {}

    template = Pipeline(
        [
            (
                "scale",
                StandardScaler(),
            ),
            (
                "pca",
                PCA(
                    24,
                    whiten=True,
                    random_state=SEED,
                ),
            ),
            (
                "lr",
                LogisticRegression(
                    C=0.1,
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=3000,
                    random_state=SEED,
                ),
            ),
        ]
    )

    for target in TARGETS:
        y = (
            gold[target]
            .astype(int)
            .to_numpy()
        )

        minimum_class = int(
            pd.Series(y)
            .value_counts()
            .min()
        )

        cv = StratifiedKFold(
            n_splits=min(
                5,
                minimum_class,
            ),
            shuffle=True,
            random_state=SEED,
        )

        oof[target] = (
            cross_val_predict(
                template,
                X_gold,
                y,
                cv=cv,
                method="predict_proba",
            )[:, 1]
        )

        fitted = (
            clone(template)
            .fit(
                X_gold,
                y,
            )
        )

        test_predictions[target] = (
            fitted
            .predict_proba(
                X_test
            )[:, 1]
        )

    return (
        oof,
        test_predictions,
    )

def main():
    print("=" * 70)
    print(
        "EXP-009 "
        "DINOv2 + Report Weak Labels"
    )
    print("=" * 70)

    data_dir = find_data()
    dino_dir = find_dino()
    weak_path = find_weak_labels()

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("Datos:", data_dir)
    print("DINO:", dino_dir)
    print("Weak labels:", weak_path)
    print("Device:", device)

    if device.type == "cuda":
        print(
            "GPU:",
            torch.cuda
            .get_device_name(0),
        )

    processor = (
        AutoImageProcessor
        .from_pretrained(
            dino_dir,
            local_files_only=True,
        )
    )

    model = (
        AutoModel
        .from_pretrained(
            dino_dir,
            local_files_only=True,
        )
        .eval()
        .to(device)
    )

    train = pd.read_csv(
        data_dir / "train.csv"
    )

    train_series = pd.read_csv(
        data_dir / "train_series.csv"
    )

    test = pd.read_csv(
        data_dir / "test.csv"
    )

    test_series = pd.read_csv(
        data_dir / "test_series.csv"
    )

    weak = pd.read_csv(
        weak_path
    )

    weak_uid_column = (
        find_uid_column(
            weak
        )
    )

    weak = weak.rename(
        columns={
            weak_uid_column: UID
        }
    )

    weak[UID] = (
        weak[UID]
        .astype(str)
    )

    train[UID] = (
        train[UID]
        .astype(str)
    )

    train_series[UID] = (
        train_series[UID]
        .astype(str)
    )

    test[UID] = (
        test[UID]
        .astype(str)
    )

    test_series[UID] = (
        test_series[UID]
        .astype(str)
    )

    score_columns = (
        find_score_columns(
            weak
        )
    )

    print("\nWeak-label columns:")

    for target in TARGETS:
        print(
            f"{target:20s} "
            f"-> "
            f"{score_columns[target]}"
        )

    gold = (
        train.loc[
            train[
                TARGETS
            ]
            .notna()
            .all(axis=1),
            [UID] + TARGETS,
        ]
        .reset_index(
            drop=True
        )
    )

    gold_uids = set(
        gold[UID]
        .astype(str)
    )

    # =======================================================
    # Medir calidad del weak-label dataset
    # SOLO como auditoría.
    # =======================================================

    # =======================================================
    # Auditoría de weak labels contra los 58 gold.
    # Los 58 gold NO se usan para entrenar el weak model.
    # =======================================================

    weak_gold = (
        gold[[UID] + TARGETS]
        .merge(
            weak[
                [UID]
                + list(
                    score_columns.values()
                )
            ],
            on=UID,
            how="inner",
            suffixes=(
                "_gold",
                "_weak",
            ),
        )
    )

    weak_label_auc = {}

    for target in TARGETS:
        gold_column = (
            f"{target}_gold"
            if f"{target}_gold"
            in weak_gold.columns
            else target
        )

        source_column = (
            score_columns[target]
        )

        weak_column = (
            f"{source_column}_weak"
            if f"{source_column}_weak"
            in weak_gold.columns
            else source_column
        )

        y_true = pd.to_numeric(
            weak_gold[
                gold_column
            ],
            errors="coerce",
        )

        y_pred = pd.to_numeric(
            weak_gold[
                weak_column
            ],
            errors="coerce",
        )

        valid = (
            y_true.notna()
            & y_pred.notna()
        )

        weak_label_auc[target] = (
            roc_auc_score(
                y_true[valid]
                .astype(int),
                y_pred[valid]
                .astype(float),
            )
        )

    weak_label_macro = float(
        np.mean(
            list(
                weak_label_auc.values()
            )
        )
    )

    print("\n" + "=" * 70)
    print(
        "WEAK LABEL QUALITY "
        "VS 58 GOLD"
    )
    print("=" * 70)

    print(
        "Weak file:",
        weak_path.name,
    )

    print(
        "Macro weak-label AUC:",
        f"{weak_label_macro:.4f}",
    )

    for target in TARGETS:
        print(
            f"{target:20s} "
            f"{weak_label_auc[target]:.4f}"
        )

    if weak_label_macro < 0.75:
        raise RuntimeError(
            "Weak labels demasiado débiles. "
            "No continuar."
        )
        
    # =======================================================
    # Excluir por completo los 58 GOLD
    # del weak training.
    # =======================================================

    weak_train = (
        train[[UID]]
        .merge(
            weak,
            on=UID,
            how="inner",
        )
    )

    weak_train = (
        weak_train[
            ~weak_train[UID]
            .isin(
                gold_uids
            )
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        "\nGold studies:",
        len(gold),
    )

    print(
        "Weak train studies:",
        len(weak_train),
    )

    print(
        "Visible test:",
        len(test),
    )

    # =======================================================
    # DINO features
    # =======================================================

    needed_train_uids = set(
        weak_train[UID]
    ) | set(
        gold[UID]
    )

    selected_train = (
        choose_series(
            train_series[
                train_series[UID]
                .isin(
                    needed_train_uids
                )
            ]
        )
    )

    selected_test = (
        choose_series(
            test_series
        )
    )

    X_weak, weak_times = (
        build_matrix(
            weak_train[UID]
            .tolist(),
            selected_train,
            data_dir
            / "train_series",
            processor,
            model,
            device,
            "WEAK",
        )
    )

    X_gold, _ = (
        build_matrix(
            gold[UID]
            .tolist(),
            selected_train,
            data_dir
            / "train_series",
            processor,
            model,
            device,
            "GOLD",
        )
    )

    X_test, test_times = (
        build_matrix(
            test[UID]
            .tolist(),
            selected_test,
            data_dir
            / "test_series",
            processor,
            model,
            device,
            "TEST",
        )
    )

    print(
        "\nShapes:",
        X_weak.shape,
        X_gold.shape,
        X_test.shape,
    )

    if weak_times:
        train_hours = (
            np.mean(weak_times)
            * len(weak_train)
            / 3600
        )

        print(
            "Weak feature time:",
            f"{train_hours:.2f} h",
        )

    if test_times:
        projected = (
            np.mean(test_times)
            * 1322
            / 3600
        )

        print(
            "Hidden test projection:",
            f"{projected:.2f} h",
        )

    # =======================================================
    # PCA fitted ONLY on weak training.
    # Gold remains genuine holdout.
    # =======================================================

    scaler = StandardScaler()

    X_weak_scaled = (
        scaler.fit_transform(
            X_weak
        )
    )

    X_gold_scaled = (
        scaler.transform(
            X_gold
        )
    )

    X_test_scaled = (
        scaler.transform(
            X_test
        )
    )

    pca = PCA(
        n_components=128,
        whiten=True,
        svd_solver="randomized",
        random_state=SEED,
    )

    Z_weak = (
        pca.fit_transform(
            X_weak_scaled
        )
    )

    Z_gold = (
        pca.transform(
            X_gold_scaled
        )
    )

    Z_test = (
        pca.transform(
            X_test_scaled
        )
    )

    print(
        "PCA shapes:",
        Z_weak.shape,
        Z_gold.shape,
        Z_test.shape,
    )

    # =======================================================
    # Candidate weak-label model families
    # =======================================================

    family_gold = {}
    family_test = {}
    family_scores = {}

    families = [
        "RIDGE20",
        "RIDGE50",
        "LOGIT",
    ]

    for family in families:
        gold_predictions = {}
        test_predictions = {}

        for target in TARGETS:
            soft_y = (
                pd.to_numeric(
                    weak_train[
                        score_columns[
                            target
                        ]
                    ],
                    errors="coerce",
                )
                .fillna(0.5)
                .to_numpy(
                    dtype=float
                )
            )

            if family == "RIDGE20":
                fitted = Ridge(
                    alpha=20.0
                )

                fitted.fit(
                    Z_weak,
                    soft_y,
                )

                gold_pred = (
                    fitted.predict(
                        Z_gold
                    )
                )

                test_pred = (
                    fitted.predict(
                        Z_test
                    )
                )

            elif family == "RIDGE50":
                fitted = Ridge(
                    alpha=50.0
                )

                fitted.fit(
                    Z_weak,
                    soft_y,
                )

                gold_pred = (
                    fitted.predict(
                        Z_gold
                    )
                )

                test_pred = (
                    fitted.predict(
                        Z_test
                    )
                )

            else:
                confidence = (
                    2.0
                    * np.abs(
                        soft_y
                        - 0.5
                    )
                )

                keep = (
                    confidence
                    >= 0.10
                )

                hard_y = (
                    soft_y[
                        keep
                    ]
                    > 0.5
                ).astype(int)

                fitted = (
                    LogisticRegression(
                        C=0.1,
                        class_weight="balanced",
                        solver="liblinear",
                        max_iter=3000,
                        random_state=SEED,
                    )
                )

                fitted.fit(
                    Z_weak[
                        keep
                    ],
                    hard_y,
                    sample_weight=(
                        confidence[
                            keep
                        ]
                    ),
                )

                gold_pred = (
                    fitted
                    .predict_proba(
                        Z_gold
                    )[:, 1]
                )

                test_pred = (
                    fitted
                    .predict_proba(
                        Z_test
                    )[:, 1]
                )

            gold_predictions[
                target
            ] = gold_pred

            test_predictions[
                target
            ] = test_pred

        macro, scores = (
            macro_auc(
                gold,
                gold_predictions,
            )
        )

        family_gold[
            family
        ] = gold_predictions

        family_test[
            family
        ] = test_predictions

        family_scores[
            family
        ] = (
            macro,
            scores,
        )

        print(
            f"\n{family} "
            f"Gold Macro AUC: "
            f"{macro:.4f}"
        )

    best_family = max(
        family_scores,
        key=lambda name:
        family_scores[name][0],
    )

    print(
        "\nBEST WEAK FAMILY:",
        best_family,
    )

    print(
        "Weak model Gold AUC:",
        f"{family_scores[best_family][0]:.4f}",
    )

    # =======================================================
    # EXP-007 gold-only arm.
    # Gold OOF avoids leakage.
    # =======================================================

    (
        gold_oof,
        gold_test_predictions,
    ) = gold_oof_and_test(
        X_gold,
        X_test,
        gold,
    )

    gold_macro, _ = macro_auc(
        gold,
        gold_oof,
    )

    print(
        "\nEXP-007 OOF arm:",
        f"{gold_macro:.4f}",
    )

    # =======================================================
    # Global rank blend:
    # weak holdout predictions
    # +
    # EXP007 gold OOF predictions.
    #
    # ONE global weight only.
    # No target-by-target overfitting.
    # =======================================================

    weak_gold = (
        family_gold[
            best_family
        ]
    )

    weak_test = (
        family_test[
            best_family
        ]
    )

    weights = [
        0.50,
        0.65,
        0.80,
        0.90,
        1.00,
    ]

    blend_results = {}

    for weak_weight in weights:
        blended_gold = {}

        for target in TARGETS:
            blended_gold[
                target
            ] = (
                weak_weight
                * rank01(
                    weak_gold[
                        target
                    ]
                )
                +
                (
                    1.0
                    - weak_weight
                )
                * rank01(
                    gold_oof[
                        target
                    ]
                )
            )

        macro, scores = (
            macro_auc(
                gold,
                blended_gold,
            )
        )

        blend_results[
            weak_weight
        ] = (
            macro,
            scores,
        )

        print(
            "Blend weak weight",
            weak_weight,
            "Macro:",
            f"{macro:.4f}",
        )

    best_weight = max(
        blend_results,
        key=lambda weight:
        blend_results[
            weight
        ][0],
    )

    final_macro = (
        blend_results[
            best_weight
        ][0]
    )

    final_scores = (
        blend_results[
            best_weight
        ][1]
    )

    print("\n" + "=" * 70)

    print(
        "BEST WEAK FAMILY:",
        best_family,
    )

    print(
        "BEST WEAK WEIGHT:",
        best_weight,
    )

    print(
        "FINAL GOLD MACRO AUC:",
        f"{final_macro:.4f}",
    )

    print("=" * 70)

    for target in TARGETS:
        print(
            f"{target:20s} "
            f"{final_scores[target]:.4f}"
        )

    # =======================================================
    # Final hidden-test prediction
    # =======================================================

    submission = (
        test[
            [UID]
        ]
        .copy()
    )

    for target in TARGETS:
        submission[target] = (
            best_weight
            * rank01(
                weak_test[
                    target
                ]
            )
            +
            (
                1.0
                - best_weight
            )
            * rank01(
                gold_test_predictions[
                    target
                ]
            )
        )

    assert (
        submission[
            TARGETS
        ]
        .notna()
        .all()
        .all()
    )

    assert (
        submission[UID]
        .is_unique
    )

    submission.to_csv(
        "/kaggle/working/"
        "submission.csv",
        index=False,
    )

    print(
        "\nSubmission creada"
    )

    print(
        submission.head()
    )


In [ ]:
"""EXP-010: mejora original, economica y verificable de EXP-009.

Se ejecuta DESPUES de cargar las funciones del codigo EXP-009, SIN llamar
EXP-009.main(). No usa reportes de test ni etiquetas gold para entrenar
el brazo weak o seleccionar epocas.

Genera: /kaggle/working/submission.csv, exp010_diagnostics.json,
exp010_gold_scores.csv. No garantiza alcanzar 0.80 Public AUC.
"""

import copy
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Variables, funciones y librerias reutilizadas desde EXP-009:
# UID, SID, TARGETS, PLANES, SEED, find_data, find_dino,
# find_weak_labels, find_uid_column, find_score_columns, choose_series,
# series_windows, encode, build_matrix, gold_oof_and_test, rank01,
# macro_auc, AutoImageProcessor, AutoModel.

OUTDIR = Path('/kaggle/working')
N_EPOCHS = 75
MLP_SEEDS = (17, 42)
BASELINE_WEAK_WEIGHT = 0.90  # Combinacion validada en EXP-009
BLEND_WEIGHTS = (0.25, 0.50, 0.75)
# Predefinimos un filtro conservador antes de observar resultados:
MIN_GOLD_MACRO = 0.790
MIN_GAIN_OVER_BASELINE = 0.035
MAX_FEATURE_HOURS = 6.4   # margen para procesamiento posterior dentro de 9h


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def one_study_exp010(study_uid, chosen, dicom_root, processor, model, device):
    """Mismo recorrido/encoder EXP009; mayor estadistica sin nuevos forward-passes.

    Primer bloque de 6*hidden es IDENTICO al EXP009 (mean,max por plano).
    Segundo bloque agrega std y mediana por plano, sin costo GPU adicional.
    """
    hidden = int(model.config.hidden_size)
    base, extras = [], []
    for plane in PLANES:
        images = []
        for series_uid in chosen.get(str(study_uid), {}).get(plane, []):
            folder = dicom_root / str(study_uid) / str(series_uid)
            if folder.exists():
                images.extend(series_windows(folder))
        features = encode(images, processor, model, device)
        if len(features):
            # Orden exactamente igual al EXP-009.
            base.extend([features.mean(axis=0), features.max(axis=0)])
            extras.extend([features.std(axis=0), np.median(features, axis=0)])
        else:
            zero = np.zeros(hidden, np.float32)
            base.extend([zero, zero.copy()])
            extras.extend([zero.copy(), zero.copy()])
    return np.concatenate(base + extras).astype(np.float32)


class MultiTaskHead(nn.Module):
    """No modifica ni copia pesos de otro competidor; aprende sobre DINOv2."""
    def __init__(self, n_features, n_targets):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(n_features),
            nn.Linear(n_features, 384),
            nn.GELU(),
            nn.Dropout(0.35),
            nn.Linear(384, 192),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(192, n_targets),
        )

    def forward(self, X):
        return self.layers(X)


def train_multitask_weak(Z, Y, Z_gold, Z_test, seed, device):
    """Selecciona epoch SOLO en 15% de weak reports; nunca en gold."""
    set_seed(seed)
    all_indices = np.arange(len(Z))
    tr_idx, va_idx = train_test_split(
        all_indices, test_size=0.15, random_state=seed, shuffle=True)

    X_tensor = torch.tensor(Z, dtype=torch.float32)
    Y_tensor = torch.tensor(Y, dtype=torch.float32)
    net = MultiTaskHead(Z.shape[1], Y.shape[1]).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=0.00065, weight_decay=0.035)
    criterion = nn.BCEWithLogitsLoss()

    X_val = X_tensor[va_idx].to(device)
    Y_val = Y_tensor[va_idx].to(device)
    best_state = None
    best_loss = float('inf')
    best_epoch = 0
    patience = 14

    for epoch in range(1, N_EPOCHS + 1):
        net.train()
        rng = np.random.default_rng(seed * 1000 + epoch)
        perm = rng.permutation(tr_idx)
        for start in range(0, len(perm), 128):
            batch_idx = perm[start:start + 128]
            X_batch = X_tensor[batch_idx].to(device)
            Y_batch = Y_tensor[batch_idx].to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = net(X_batch)
            loss = criterion(logits, Y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 2.0)
            optimizer.step()

        net.eval()
        with torch.no_grad():
            val_loss = float(criterion(net(X_val), Y_val).item())
        if val_loss < best_loss - 0.00004:
            best_state = copy.deepcopy(net.state_dict())
            best_loss = val_loss
            best_epoch = epoch
        if (epoch % 10 == 0 or epoch == 1):
            print(f'EXP010 seed={seed}, epoch={epoch}, weak-val BCE={val_loss:.5f}', flush=True)
        if epoch - best_epoch >= patience:
            break

    assert best_state is not None
    net.load_state_dict(best_state)
    net.eval()

    def predict(X):
        outputs = []
        with torch.no_grad():
            for start in range(0, len(X), 512):
                tensor = torch.tensor(X[start:start + 512], dtype=torch.float32, device=device)
                outputs.append(torch.sigmoid(net(tensor)).cpu().numpy())
        return np.concatenate(outputs, axis=0)

    print(f'EXP010 seed={seed}: mejor epoch={best_epoch}, weak-val BCE={best_loss:.5f}', flush=True)
    return predict(Z_gold), predict(Z_test), best_loss, best_epoch


def soft_matrix(weak_train, score_columns):
    columns = []
    for target in TARGETS:
        values = pd.to_numeric(weak_train[score_columns[target]], errors='coerce').to_numpy(float)
        if np.isfinite(values).sum() < 100:
            raise ValueError(f'Sin etiquetas weak validas para {target}')
        midpoint = float(np.nanmedian(values))
        values = np.where(np.isfinite(values), values, midpoint)
        if float(np.min(values)) < 0 or float(np.max(values)) > 1:
            raise ValueError(f'Etiquetas weak fuera de [0,1]: {target}')
        columns.append(values)
    return np.column_stack(columns).astype(np.float32)


def baseline_exp009(Z_weak, Z_gold, Z_test, gold, X_gold_base, X_test_base, Y):
    """Reconstruye exactamente la configuracion publica de EXP-009."""
    weak_gold = {}
    weak_test = {}
    for col, target in enumerate(TARGETS):
        soft = Y[:, col]
        confidence = 2.0 * np.abs(soft - 0.5)
        keep = confidence >= 0.10
        labels = (soft[keep] > 0.5).astype(int)
        if len(np.unique(labels)) < 2:
            raise ValueError(f'Solo una clase weak: {target}')
        clf = LogisticRegression(
            C=0.1, class_weight='balanced', solver='liblinear',
            max_iter=3000, random_state=SEED,
        )
        clf.fit(Z_weak[keep], labels, sample_weight=confidence[keep])
        weak_gold[target] = clf.predict_proba(Z_gold)[:, 1]
        weak_test[target] = clf.predict_proba(Z_test)[:, 1]

    gold_oof, gold_test = gold_oof_and_test(X_gold_base, X_test_base, gold)
    baseline_gold = {}
    baseline_test = {}
    for target in TARGETS:
        baseline_gold[target] = (
            BASELINE_WEAK_WEIGHT * rank01(weak_gold[target])
            + (1 - BASELINE_WEAK_WEIGHT) * rank01(gold_oof[target])
        )
        baseline_test[target] = (
            BASELINE_WEAK_WEIGHT * rank01(weak_test[target])
            + (1 - BASELINE_WEAK_WEIGHT) * rank01(gold_test[target])
        )
    return baseline_gold, baseline_test


def validate_and_submit(gold, test, base_gold, base_test, mlp_gold, mlp_test, diagnostics):
    base_auc, base_scores = macro_auc(gold, base_gold)
    print('\nEXP-009 BASELINE GOLD MACRO AUC:', f'{base_auc:.4f}', flush=True)
    # Este chequeo evita que un cambio silencioso rompa la reproduccion de EXP009.
    if abs(base_auc - 0.7461) > 0.012:
        raise RuntimeError(
            f'Baseline EXP009 alterado: {base_auc:.4f} vs 0.7461. '
            'NO enviar. Revisar pixel pipeline y labels.'
        )

    if not all(np.isfinite(v).all() for v in mlp_gold.values()):
        raise RuntimeError('Predicciones MLP no finitas')

    candidate_scores = {}
    for w in BLEND_WEIGHTS:
        combined_gold = {
            t: (1.0 - w) * base_gold[t] + w * rank01(mlp_gold[t])
            for t in TARGETS
        }
        candidate_scores[w] = macro_auc(gold, combined_gold)
        print(f'EXP010 BLEND MLP={w:.2f} GOLD MACRO AUC: {candidate_scores[w][0]:.4f}', flush=True)

    winner = max(candidate_scores, key=lambda w: candidate_scores[w][0])
    improved_auc, improved_targets = candidate_scores[winner]
    gate = improved_auc >= MIN_GOLD_MACRO and improved_auc - base_auc >= MIN_GAIN_OVER_BASELINE

    diagnostics.update({
        'baseline_gold_auc': float(base_auc),
        'candidate_gold_auc': float(improved_auc),
        'candidate_mlp_weight': float(winner),
        'candidate_gold_gain': float(improved_auc - base_auc),
        'min_gold_macro_to_submit': MIN_GOLD_MACRO,
        'min_gain_to_submit': MIN_GAIN_OVER_BASELINE,
        'recommended_to_submit': bool(gate),
    })

    print('\n' + '=' * 70)
    print('EXP010 GOLD VALIDATION FINAL')
    print('EXP009 GOLD BASELINE:', f'{base_auc:.4f}')
    print('EXP010 BEST GOLD:', f'{improved_auc:.4f}')
    print('BEST MLP WEIGHT:', winner)
    print('RECOMENDACION SUBMIT:', 'SI' if gate else 'NO')
    print('Ningun score local garantiza 0.800 publico.')
    print('=' * 70)

    score_rows = []
    for t in TARGETS:
        score_rows.append({
            'target': t,
            'exp009_gold_auc': float(base_scores[t]),
            'exp010_gold_auc': float(improved_targets[t]),
            'delta': float(improved_targets[t] - base_scores[t]),
        })
    pd.DataFrame(score_rows).to_csv(OUTDIR / 'exp010_gold_scores.csv', index=False)

    result = test[[UID]].copy()
    for t in TARGETS:
        if gate:
            result[t] = (1 - winner) * base_test[t] + winner * rank01(mlp_test[t])
        else:
            # Fallback para que incluso una version de prueba tenga CSV valido.
            # NO ENVIAR si el gate indica NO; ya tienes EXP009=0.762.
            result[t] = base_test[t]

    assert len(result) == len(test), 'Faltan estudios test'
    assert result[UID].is_unique
    assert result[TARGETS].notna().all().all()
    assert np.isfinite(result[TARGETS].to_numpy(float)).all()
    assert (result[TARGETS].to_numpy(float) >= 0).all()
    assert (result[TARGETS].to_numpy(float) <= 1).all()
    result.to_csv(OUTDIR / 'submission.csv', index=False)
    (OUTDIR / 'exp010_diagnostics.json').write_text(
        json.dumps(diagnostics, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    print('Submission guardada:', OUTDIR / 'submission.csv', 'Filas:', len(result))
    return gate


def main010():
    start = time.time()
    set_seed(SEED)
    print('=' * 70)
    print('EXP-010 | DINOv2 features enriquecidos + MLP multietiqueta')
    print('Baseline protegido: EXP-009 0.762 Public')
    print('=' * 70)
    if not torch.cuda.is_available():
        raise RuntimeError('Activa GPU T4 en Kaggle antes de ejecutar EXP010.')

    data_dir = find_data()
    dino_dir = find_dino()
    weak_path = find_weak_labels()
    device = torch.device('cuda')
    print('Dataset:', data_dir, 'DINO:', dino_dir, 'Labels:', weak_path)
    processor = AutoImageProcessor.from_pretrained(dino_dir, local_files_only=True)
    model = AutoModel.from_pretrained(dino_dir, local_files_only=True).eval().to(device)

    train = pd.read_csv(data_dir / 'train.csv')
    test = pd.read_csv(data_dir / 'test.csv')
    train_series = pd.read_csv(data_dir / 'train_series.csv')
    test_series = pd.read_csv(data_dir / 'test_series.csv')
    weak = pd.read_csv(weak_path)
    weak = weak.rename(columns={find_uid_column(weak): UID})
    for d in [train, test, train_series, test_series, weak]:
        d[UID] = d[UID].astype(str)
    gold = train.loc[train[TARGETS].notna().all(axis=1), [UID] + TARGETS].reset_index(drop=True)
    gold_uids = set(gold[UID])
    weak_train = train[[UID]].merge(weak, on=UID, how='inner')
    weak_train = weak_train.loc[~weak_train[UID].isin(gold_uids)].reset_index(drop=True)
    score_columns = find_score_columns(weak)
    Y = soft_matrix(weak_train, score_columns)
    print('GOLD:', len(gold), 'WEAK:', len(weak_train), 'TEST:', len(test), flush=True)
    if len(gold) != 58 or len(weak_train) < 4000:
        raise RuntimeError('Dataset TRAIN inesperado; no iniciar procesamiento largo')

    selected_train = choose_series(train_series)
    selected_test = choose_series(test_series)
    # build_matrix usa la funcion global one_study: la sustituimos temporalmente
    # para NO ejecutar DINO dos veces por estudio.
    global one_study
    old_one_study = one_study
    one_study = one_study_exp010
    try:
        Xw, times_w = build_matrix(
            weak_train[UID].tolist(), selected_train, data_dir / 'train_series',
            processor, model, device, 'WEAK010'
        )
        Xg, _ = build_matrix(
            gold[UID].tolist(), selected_train, data_dir / 'train_series',
            processor, model, device, 'GOLD010'
        )
        Xt, times_t = build_matrix(
            test[UID].tolist(), selected_test, data_dir / 'test_series',
            processor, model, device, 'TEST010'
        )
    finally:
        one_study = old_one_study

    # Liberar GPU usada para extraccion; luego MLP usa GPU con poca memoria.
    del model
    torch.cuda.empty_cache()
    print('EXTENDED SHAPES:', Xw.shape, Xg.shape, Xt.shape)
    base_dim = len(PLANES) * 2 * 384  # DINOv2-Small hidden=384
    if Xw.shape[1] != base_dim * 2:
        raise RuntimeError(f'Tamaño de DINOv2 inesperado: {Xw.shape[1]}')
    estimated_h = np.mean(times_w) * len(weak_train) / 3600 + np.mean(times_t) * len(test) / 3600
    hidden_h = np.mean(times_w) * len(weak_train) / 3600 + np.mean(times_t) * 1322 / 3600
    print(f'FEATURES draft={estimated_h:.2f} h; hidden approx={hidden_h:.2f} h')
    if hidden_h > MAX_FEATURE_HOURS:
        raise RuntimeError('Tiempo estimado de features muy largo para límite 9h. No enviar.')

    # --- Reproducir exactamente la representacion de EXP009.
    Xw_base, Xg_base, Xt_base = Xw[:, :base_dim], Xg[:, :base_dim], Xt[:, :base_dim]
    scaler = StandardScaler()
    Zw = scaler.fit_transform(Xw_base)
    Zg = scaler.transform(Xg_base)
    Zt = scaler.transform(Xt_base)
    pca_base = PCA(n_components=128, whiten=True, svd_solver='randomized', random_state=SEED)
    Zw_base = pca_base.fit_transform(Zw)
    Zg_base, Zt_base = pca_base.transform(Zg), pca_base.transform(Zt)

    base_gold, base_test = baseline_exp009(
        Zw_base, Zg_base, Zt_base, gold, Xg_base, Xt_base, Y
    )

    # --- Nuevo brazo: medianas/desviaciones de cada plano + red multi-output.
    ext_scaler = StandardScaler()
    Zw_ext = ext_scaler.fit_transform(Xw)
    Zg_ext = ext_scaler.transform(Xg)
    Zt_ext = ext_scaler.transform(Xt)
    pca_ext = PCA(n_components=256, whiten=True, svd_solver='randomized', random_state=SEED)
    Zw_ext = pca_ext.fit_transform(Zw_ext).astype(np.float32)
    Zg_ext = pca_ext.transform(Zg_ext).astype(np.float32)
    Zt_ext = pca_ext.transform(Zt_ext).astype(np.float32)

    predictions_g = []
    predictions_t = []
    training_log = []
    for seed in MLP_SEEDS:
        out_g, out_t, val_bce, best_epoch = train_multitask_weak(
            Zw_ext, Y, Zg_ext, Zt_ext, seed, device
        )
        predictions_g.append(out_g)
        predictions_t.append(out_t)
        training_log.append({'seed': seed, 'weak_val_bce': val_bce, 'best_epoch': best_epoch})
    pg = np.mean(predictions_g, axis=0)
    pt = np.mean(predictions_t, axis=0)
    mlp_gold = {t: pg[:, i] for i, t in enumerate(TARGETS)}
    mlp_test = {t: pt[:, i] for i, t in enumerate(TARGETS)}
    diagnostics = {
        'model': 'EXP010 original multitask-MLP on DINOv2 stats',
        'weak_train_studies': len(weak_train), 'gold_studies': len(gold),
        'test_studies': len(test), 'weak_labels': weak_path.name,
        'feature_hours_estimate_hidden': float(hidden_h),
        'multitask_weak_validation': training_log,
    }
    gate = validate_and_submit(gold, test, base_gold, base_test, mlp_gold, mlp_test, diagnostics)
    print(f'EXP010 COMPLETO en {(time.time()-start)/3600:.2f} horas', flush=True)
    print('SI SUBMIT:', bool(gate), flush=True)


if __name__ == '__main__':
    main010()
